<a href="https://colab.research.google.com/github/javidfarsoft-bot/brain-tumor-detection-comparison/blob/main/notebooks/02_baseline_fasterrcnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 3 — Faster R-CNN Baseline

Transfer learning with `fasterrcnn_resnet50_fpn_v2` (COCO pretrained), box-aware augmentation, per-epoch validation (loss/precision/recall/mAP), and early stopping on mAP@0.5 (chosen over val_loss because Faster R-CNN's internal RPN/ROI sampling makes val_loss noisy — see report for details).

**Prerequisite:** run `notebooks/01_dataset_exploration.ipynb` first (or its download + `merge_dataset.py` cells) so that `data/yolo/{train,valid,test}` exists.

In [ ]:
import torch
print(torch.__version__, torch.cuda.is_available())

2.11.0+cpu False


In [ ]:
!pip install pycocotools torchmetrics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 4.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/brain-tumor-project/checkpoints', exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!git clone https://github.com/javidfarsoft-bot/brain-tumor-detection-comparison.git
%cd brain-tumor-detection-comparison
!pip install kaggle pycocotools torchmetrics -q

Cloning into 'brain-tumor-detection-comparison'...
remote: Enumerating objects: 163, done.
remote: Counting objects: 100% (163/163), done.
remote: Compressing objects: 100% (122/122), done.
remote: Total 163 (delta 65), reused 127 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (163/163), 17.62 MiB | 26.34 MiB/s, done.
Resolving deltas: 100% (65/65), done.
/content/brain-tumor-detection-comparison


In [ ]:
!mkdir -p ~/.kaggle
!echo YOUR_KAGGLE_TOKEN > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token
!kaggle datasets download -d ahmedsorour1/mri-for-brain-tumor-with-bounding-boxes
!mkdir -p data/raw
!unzip -q -o mri-for-brain-tumor-with-bounding-boxes.zip -d data/raw
!python -m src.data.merge_dataset --src data/raw --dst data/yolo

Dataset URL: https://www.kaggle.com/datasets/ahmedsorour1/mri-for-brain-tumor-with-bounding-boxes
License(s): CC0-1.0
100% 133M/133M [00:06<00:00, 20.7MB/s]

Total images found: 5247
Train: 4197  Valid: 522  Test: 528
Done.


In [ ]:
import json
from pathlib import Path
from PIL import Image

def yolo_to_coco(yolo_dir, split, class_names):
    img_dir = Path(yolo_dir) / split / "images"
    lbl_dir = Path(yolo_dir) / split / "labels"
    coco = {"images": [], "annotations": [], "categories": [{"id": i, "name": n} for i, n in enumerate(class_names)]}
    ann_id = 1
    for img_id, img_path in enumerate(sorted(img_dir.iterdir()), start=1):
        with Image.open(img_path) as im:
            w, h = im.size
        coco["images"].append({"id": img_id, "file_name": img_path.name, "width": w, "height": h})
        lbl_path = lbl_dir / f"{img_path.stem}.txt"
        if not lbl_path.exists():
            continue
        for line in lbl_path.read_text().strip().splitlines():
            cls, xc, yc, bw, bh = map(float, line.split())
            box_w, box_h = bw * w, bh * h
            x_min, y_min = xc * w - box_w / 2, yc * h - box_h / 2
            coco["annotations"].append({"id": ann_id, "image_id": img_id, "category_id": int(cls),
                                         "bbox": [x_min, y_min, box_w, box_h], "area": box_w * box_h, "iscrowd": 0})
            ann_id += 1
    out_dir = Path("data/coco")
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / f"{split}.json").write_text(json.dumps(coco))
    print(f"{split}: {len(coco['images'])} images, {len(coco['annotations'])} annotations")

class_names = ["Glioma", "Meningioma", "Pituitary", "No Tumor"]
for split in ["train", "valid", "test"]:
    yolo_to_coco("data/yolo", split, class_names)

train: 4197 images, 4711 annotations
valid: 522 images, 577 annotations
test: 528 images, 583 annotations


In [ ]:
from torch.utils.data import Dataset, DataLoader
from pycocotools.coco import COCO
import torchvision.transforms.v2 as T
from torchvision import tv_tensors
from pathlib import Path
from PIL import Image

def get_transforms(train: bool):
    if train:
        return T.Compose([
            T.RandomHorizontalFlip(p=0.5),
            T.RandomRotation(degrees=10),
            T.ColorJitter(brightness=0.2, contrast=0.2),
            T.ToDtype(torch.float32, scale=True),
        ])
    return T.Compose([T.ToDtype(torch.float32, scale=True)])

class CocoDataset(Dataset):
    def __init__(self, images_dir, ann_file, train=True):
        self.images_dir = Path(images_dir)
        self.coco = COCO(ann_file)
        self.ids = list(sorted(self.coco.imgs.keys()))
        self.transforms = get_transforms(train)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        info = self.coco.imgs[img_id]
        img = Image.open(self.images_dir / info["file_name"]).convert("RGB")
        img = tv_tensors.Image(img)
        anns = self.coco.loadAnns(self.coco.getAnnIds(imgIds=img_id))
        boxes, labels = [], []
        for ann in anns:
            x, y, w, h = ann["bbox"]
            boxes.append([x, y, x + w, y + h])
            labels.append(ann["category_id"] + 1)
        if boxes:
            boxes = tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=(info["height"], info["width"]))
            labels = torch.tensor(labels, dtype=torch.int64)
        else:
            boxes = tv_tensors.BoundingBoxes(torch.zeros((0, 4)), format="XYXY", canvas_size=(info["height"], info["width"]))
            labels = torch.zeros((0,), dtype=torch.int64)
        img, boxes = self.transforms(img, boxes)
        return img, {"boxes": boxes, "labels": labels, "image_id": torch.tensor([img_id])}

def collate_fn(batch):
    return tuple(zip(*batch))

class_names = ["Glioma", "Meningioma", "Pituitary", "No Tumor"]
train_ds = CocoDataset("data/yolo/train/images", "data/coco/train.json", train=True)
valid_ds = CocoDataset("data/yolo/valid/images", "data/coco/valid.json", train=False)
test_ds  = CocoDataset("data/yolo/test/images",  "data/coco/test.json",  train=False)

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, collate_fn=collate_fn, num_workers=2)
valid_loader = DataLoader(valid_ds, batch_size=2, shuffle=False, collate_fn=collate_fn, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=2, shuffle=False, collate_fn=collate_fn, num_workers=2)

loading annotations into memory...
Done (t=0.04s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!


In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(class_names) + 1
model = fasterrcnn_resnet50_fpn_v2(weights="DEFAULT")
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
model.to(device)

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth


100%|██████████| 167M/167M [00:01<00:00, 101MB/s] 


FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
       

In [ ]:
def box_iou(box_a, box_b):
    area_a = (box_a[:, 2] - box_a[:, 0]) * (box_a[:, 3] - box_a[:, 1])
    area_b = (box_b[:, 2] - box_b[:, 0]) * (box_b[:, 3] - box_b[:, 1])
    lt = torch.max(box_a[:, None, :2], box_b[None, :, :2])
    rb = torch.min(box_a[:, None, 2:], box_b[None, :, 2:])
    wh = (rb - lt).clamp(min=0)
    inter = wh[:, :, 0] * wh[:, :, 1]
    union = area_a[:, None] + area_b[None, :] - inter
    return inter / union.clamp(min=1e-6)

def compute_precision_recall(preds, targets, iou_threshold=0.5, score_threshold=0.5):
    tp, fp, fn = 0, 0, 0
    for pred, tgt in zip(preds, targets):
        keep = pred["scores"] >= score_threshold
        pred_boxes = pred["boxes"][keep]
        if len(pred_boxes) == 0 and len(tgt["boxes"]) == 0:
            continue
        if len(pred_boxes) == 0:
            fn += len(tgt["boxes"]); continue
        if len(tgt["boxes"]) == 0:
            fp += len(pred_boxes); continue
        ious = box_iou(pred_boxes, tgt["boxes"])
        matched = set()
        for i in range(len(pred_boxes)):
            best_iou, best_j = ious[i].max(0)
            if best_iou >= iou_threshold and best_j.item() not in matched:
                tp += 1; matched.add(best_j.item())
            else:
                fp += 1
        fn += len(tgt["boxes"]) - len(matched)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return precision, recall

In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torch.cuda.amp import autocast, GradScaler

@torch.no_grad()
def evaluate_epoch(model, loader, device):
    model.train()
    total_loss = 0.0
    for images, targets in loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        with autocast():
            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())
        total_loss += loss.item()
    val_loss = total_loss / len(loader)

    model.eval()
    metric = MeanAveragePrecision(iou_type="bbox")
    all_preds, all_targets = [], []
    for images, targets in loader:
        images = [img.to(device) for img in images]
        outputs = model(images)
        preds = [{"boxes": o["boxes"].cpu(), "scores": o["scores"].cpu(), "labels": o["labels"].cpu()} for o in outputs]
        tgts = [{"boxes": t["boxes"].cpu(), "labels": t["labels"].cpu()} for t in targets]
        metric.update(preds, tgts)
        all_preds.extend(preds); all_targets.extend(tgts)
    result = metric.compute()
    precision, recall_pr = compute_precision_recall(all_preds, all_targets)
    return {"val_loss": val_loss, "mAP_50": result["map_50"].item(),
            "mAP_50_95": result["map"].item(), "precision": precision, "recall": recall_pr}

## Training

Early stopping on mAP@0.5 (patience=15). Ran for 29 epochs; best model at epoch 14 (mAP@0.5 = 0.9527). Checkpoints saved to Google Drive each epoch for resumability across sessions (Colab free-tier GPU quota required multiple sessions to complete this run).

In [ ]:
import json
with open("results/tables/fasterrcnn_history.json") as f:
    history = json.load(f)
print(f"Epochs recorded: {len(history['train_loss'])}")
print(f"Best mAP@0.5: {max(history['mAP_50']):.4f} at epoch {history['mAP_50'].index(max(history['mAP_50']))+1}")

Epochs recorded: 29
Best mAP@0.5: 0.9527 at epoch 14


## Results

![Training curves](https://github.com/javidfarsoft-bot/brain-tumor-detection-comparison/blob/main/results/figures/fasterrcnn_training_curves.png?raw=1)

| Model | Best Epoch | Precision | Recall | mAP@0.5 | mAP@0.5:0.95 | Params (M) | GFLOPs | FPS | Model size (MB) |
|---|---|---|---|---|---|---|---|---|---|
| Faster R-CNN | 14 | 0.8366 | 0.9526 | 0.9527 | 0.6711 | 43.27 | 280.81 | 7.15 | 329.69 |

Full table: `results/tables/fasterrcnn_baseline_summary.csv`